In [1]:
import numpy as np
import pandas as pd
import pickle

# %% dataset settings -------------------------------------------------------------
datasets = ["ERA5", "MERRA2", "JRA3Q"]

OUT_DIR_List = {
    "ERA5": "/scratch/bell/hu1029/LGHW/interm_ERA5",
    "MERRA2": "/scratch/bell/hu1029/LGHW/interm_MERRA2",
    "JRA3Q": "/scratch/bell/hu1029/LGHW/interm_JRA3Q"
}
yearnameList = {
    "ERA5": "1979_2025",
    "MERRA2": "1980_2025",
    "JRA3Q": "1979_2025"
}

regions = ["ATL", "NP", "SP"]
ss = "ALL"
blkTypes = ["Ridge", "Trough", "Dipole"]

rows = []


In [2]:

for cyc in ['AC', 'CC']:
    for dtname in datasets:
        
        OUTDIR = OUT_DIR_List[dtname]
        yearname = yearnameList[dtname]

        for rgname in regions:
            for typeid in [1, 2, 3]:

                # get the blocking index each track contribute to (-1 or the blocking global index), 1d, same length as the track_data
                InteractingBlockID = np.load(f'{OUTDIR}/{dtname}_TrackBlockingType{typeid}_Index_{yearname}_{rgname}_{ss}_{cyc}.npy')        
                # get the blocking event index list (global index), 1d list, all blocking events in the target region
                with open(f'{OUTDIR}/{dtname}_SD_BlockingFlagmaskClustersEventList_Type{typeid}_{rgname}_{ss}', "rb") as f:
                    Sec2BlockEvent = pickle.load(f)

                # 01 the number of blocking events within the target region --------------------------------
                numBlockEvent = len(Sec2BlockEvent)
                InteractingBlockID_uniquev = InteractingBlockID[InteractingBlockID != -1] # remove the -1
                InteractingBlockID_uniquev = np.unique(InteractingBlockID_uniquev) # get the unique values
                BlockwithTrackLen = len(InteractingBlockID_uniquev) # the number of unique blocking events that has ever interact with the tracks

                rows.append({
                    "EddyType": cyc,
                    "Dataset": dtname,
                    "Type": blkTypes[typeid - 1],
                    "Region": rgname,
                    "Number": numBlockEvent,
                    "BLKwithTrackNumber": BlockwithTrackLen,
                    "BLKwithTrackRatio": BlockwithTrackLen/numBlockEvent
                })

df = pd.DataFrame(rows)
print(df.head())


  EddyType Dataset    Type Region  Number  BLKwithTrackNumber  \
0       AC    ERA5   Ridge    ATL     702                 627   
1       AC    ERA5  Trough    ATL     264                  70   
2       AC    ERA5  Dipole    ATL     122                  95   
3       AC    ERA5   Ridge     NP     211                 187   
4       AC    ERA5  Trough     NP     733                 343   

   BLKwithTrackRatio  
0           0.893162  
1           0.265152  
2           0.778689  
3           0.886256  
4           0.467940  


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

df["Dataset"] = pd.Categorical(
    df["Dataset"],
    categories=["ERA5", "MERRA2", "JRA3Q"],
    ordered=True
)

df["Type"] = pd.Categorical(
    df["Type"],
    categories=["Ridge", "Trough", "Dipole"],
    ordered=True
)

# pivot
df_S1 = df.pivot_table(
    index=["Dataset", "Type"],
    columns="Region",
    values="Number",
    aggfunc="first"
).sort_index()
df_S1 = df_S1.astype(int)
print(df_S1)

# latex
print(df_S1.to_latex(multirow=True))
df_S1.to_csv("table_S1.csv")

Region          ATL   NP   SP
Dataset Type                 
ERA5    Ridge   702  211  631
        Trough  264  733  185
        Dipole  122  124   59
MERRA2  Ridge   685  222  629
        Trough  270  710  185
        Dipole  115  129   65
JRA3Q   Ridge   702  228  639
        Trough  264  716  173
        Dipole  124  125   73
\begin{tabular}{llrrr}
\toprule
 & Region & ATL & NP & SP \\
Dataset & Type &  &  &  \\
\midrule
\multirow[t]{3}{*}{ERA5} & Ridge & 702 & 211 & 631 \\
 & Trough & 264 & 733 & 185 \\
 & Dipole & 122 & 124 & 59 \\
\cline{1-5}
\multirow[t]{3}{*}{MERRA2} & Ridge & 685 & 222 & 629 \\
 & Trough & 270 & 710 & 185 \\
 & Dipole & 115 & 129 & 65 \\
\cline{1-5}
\multirow[t]{3}{*}{JRA3Q} & Ridge & 702 & 228 & 639 \\
 & Trough & 264 & 716 & 173 \\
 & Dipole & 124 & 125 & 73 \\
\cline{1-5}
\bottomrule
\end{tabular}



/tmp/ipykernel_3818427/566219043.py:18: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_S1 = df.pivot_table(


In [8]:
import pandas as pd

def make_table_S2(df, cyc):

    # AC or CC
    df_sub = df[df["EddyType"] == cyc].copy()

    # categorical
    df_sub["Dataset"] = pd.Categorical(
        df_sub["Dataset"],
        categories=["ERA5", "MERRA2", "JRA3Q"],
        ordered=True
    )

    df_sub["Type"] = pd.Categorical(
        df_sub["Type"],
        categories=["Ridge", "Trough", "Dipole"],
        ordered=True
    )

    df_sub["Region"] = pd.Categorical(
        df_sub["Region"],
        categories=["ATL", "NP", "SP"],
        ordered=True
    )

    # pivot
    df_num = df_sub.pivot_table(
        index=["Dataset", "Type"],
        columns="Region",
        values="BLKwithTrackNumber",
        aggfunc="first",
        observed=False
    ).sort_index()

    df_ratio = df_sub.pivot_table(
        index=["Dataset", "Type"],
        columns="Region",
        values="BLKwithTrackRatio",
        aggfunc="first",
        observed=False
    ).sort_index()

    # select and order columns
    df_num = df_num[["ATL", "NP", "SP"]]
    df_ratio = df_ratio[["ATL", "NP", "SP"]]

    # combine
    df_S2 = df_num.copy()

    for col in ["ATL", "NP", "SP"]:
        df_S2[col] = (
            df_num[col].astype(int).astype(str)
            + " ("
            + (df_ratio[col] * 100).round(1).map(lambda x: f"{x:.1f}")
            + "%)"
        )

    # average
    df_S2["Average"] = (df_ratio.mean(axis=1) * 100).round(1).map(lambda x: f"{x:.1f}%")

    return df_S2

df = pd.DataFrame(rows)

df_S2_AC = make_table_S2(df, "AC")
df_S2_CC = make_table_S2(df, "CC")

print("=== AC ===")
print(df_S2_AC)
df_S2_AC.to_csv("table_S2_AC.csv")

print("=== CC ===")
print(df_S2_CC)
df_S2_CC.to_csv("table_S2_CC.csv")

=== AC ===
Region                  ATL           NP           SP Average
Dataset Type                                                 
ERA5    Ridge   627 (89.3%)  187 (88.6%)  563 (89.2%)   89.1%
        Trough   70 (26.5%)  343 (46.8%)   42 (22.7%)   32.0%
        Dipole   95 (77.9%)   88 (71.0%)   52 (88.1%)   79.0%
MERRA2  Ridge   603 (88.0%)  195 (87.8%)  566 (90.0%)   88.6%
        Trough   67 (24.8%)  307 (43.2%)   43 (23.2%)   30.4%
        Dipole   93 (80.9%)   95 (73.6%)   59 (90.8%)   81.8%
JRA3Q   Ridge   620 (88.3%)  207 (90.8%)  571 (89.4%)   89.5%
        Trough   74 (28.0%)  336 (46.9%)   39 (22.5%)   32.5%
        Dipole   91 (73.4%)   96 (76.8%)   64 (87.7%)   79.3%
=== CC ===
Region                  ATL           NP           SP Average
Dataset Type                                                 
ERA5    Ridge   348 (49.6%)  103 (48.8%)  268 (42.5%)   47.0%
        Trough  238 (90.2%)  675 (92.1%)  176 (95.1%)   92.5%
        Dipole  108 (88.5%)  107 (86.3%)   50 (8